<a href="https://colab.research.google.com/github/senohmbi-debug/ITCS-3162/blob/main/lab_03_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — Example: Data Wrangling with the Diamonds Dataset

**ITCS 3162 — Introduction to Data Mining**
**Companion to Zybooks Chapter 3 (Data Wrangling)**

Real-world data is rarely clean. "Wrangling" is the unglamorous-but-essential work of getting raw data into a shape you can actually analyze — fixing types, handling missing values, removing duplicates, and reshaping rows and columns.

This walkthrough uses the **Diamonds dataset** (the same one your Zybooks chapter uses): roughly 54,000 diamonds with their carat, cut, color, clarity, dimensions, and price.

## Learning objectives
By the end you will be able to:
1. Load a dataset with `pandas` from a built-in source or a CSV
2. Inspect data types, shape, and basic structure with `info()`, `head()`, `shape`, `dtypes`
3. Find and handle missing values
4. Detect and remove duplicates
5. Filter rows and select columns
6. Create derived columns (feature engineering preview)
7. Group and summarize with `groupby()`


## 1. Load the data

The diamonds dataset ships with seaborn — no download needed. Run the cell below.


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset("diamonds")

# Ensure ordered category dtypes for the three quality columns. seaborn already
# does this, but doing it explicitly makes the code portable to a plain CSV load
# and teaches you about ordered categoricals.
df["cut"]     = pd.Categorical(df["cut"],
                  categories=["Fair","Good","Very Good","Premium","Ideal"], ordered=True)
df["color"]   = pd.Categorical(df["color"],
                  categories=list("JIHGFED"), ordered=True)  # J=worst, D=best
df["clarity"] = pd.Categorical(df["clarity"],
                  categories=["I1","SI2","SI1","VS2","VS1","VVS2","VVS1","IF"], ordered=True)

df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


### Column dictionary

| Column | Type | Meaning |
|---|---|---|
| `carat` | float | Weight of the diamond |
| `cut`   | categorical | Quality of the cut (Fair < Good < Very Good < Premium < Ideal) |
| `color` | categorical | D (best) through J (worst) |
| `clarity` | categorical | I1 (worst) through IF (best) |
| `depth` | float | Total depth percentage |
| `table` | float | Width of top relative to widest point |
| `price` | int | Price in USD |
| `x`, `y`, `z` | float | Length, width, depth in mm |


## 2. First-look inspection

Three commands you'll run on *every* new dataset:


In [ ]:
print("Shape (rows, cols):", df.shape)
print("Memory used:", df.memory_usage(deep=True).sum() / 1024**2, "MB")

In [ ]:
df.info()

`info()` tells you the column types and how many non-null entries each has. `category` is a memory-efficient dtype for repeated strings — it's like an enum.


In [ ]:
df.describe()

`describe()` summarizes numeric columns. Notice the strange minimums for `x`, `y`, `z` — diamonds with zero dimensions are physically impossible, hinting at data quality issues we'll investigate.

To summarize the categorical columns separately:


In [ ]:
df.describe(include="category")

## 3. Missing values

In this dataset there aren't any nulls, but checking is the habit:


In [ ]:
df.isna().sum()

When you *do* find missing values, the three usual options are:
- **Drop** rows: `df.dropna()`
- **Drop** columns that are mostly missing: `df.drop(columns=["bad_col"])`
- **Fill** with a sensible value: `df["age"].fillna(df["age"].median())`

The right choice depends on *why* the values are missing (you'll see this in Lab 5).


## 4. Duplicates

Are there any exact duplicates?


In [ ]:
dups = df.duplicated().sum()
print(f"Duplicate rows: {dups}")

# Remove them (returns a new DataFrame; original is unchanged)
df_dedup = df.drop_duplicates()
print(f"After dedup: {df_dedup.shape}")

## 5. Data quality: impossible values

Earlier `describe()` showed `x`, `y`, `z` minimums of zero. Let's find those rows.


In [ ]:
bad = df[(df["x"] == 0) | (df["y"] == 0) | (df["z"] == 0)]
print(f"Rows with a zero dimension: {len(bad)}")
bad.head()

These are almost certainly measurement errors. A reasonable wrangling step is to drop them.


In [ ]:
df_clean = df[(df["x"] > 0) & (df["y"] > 0) & (df["z"] > 0)].copy()
print(f"Before: {len(df)} rows | After: {len(df_clean)} rows | Removed: {len(df) - len(df_clean)}")

> **Why `.copy()`?** Without it, `df_clean` would be a *view* into `df`, and modifying it later would trigger pandas' `SettingWithCopyWarning`. Use `.copy()` whenever you take a subset you intend to modify.


## 6. Filtering and selecting

Pull out only the columns you care about, only the rows that matter.


In [ ]:
# Just two columns
df_clean[["carat", "price"]].head()

In [ ]:
# Boolean filtering — "Ideal cut, larger than 1 carat"
mask = (df_clean["cut"] == "Ideal") & (df_clean["carat"] > 1.0)
ideal_big = df_clean[mask]
print(f"Ideal-cut diamonds over 1 carat: {len(ideal_big)}")
ideal_big.head()

## 7. Creating new columns (feature engineering preview)

A derived feature can be more useful than the raw inputs. Let's add **price per carat** (a more comparable measure of value) and a **volume** column.


In [ ]:
df_clean["price_per_carat"] = df_clean["price"] / df_clean["carat"]
df_clean["volume_mm3"] = df_clean["x"] * df_clean["y"] * df_clean["z"]

df_clean[["carat", "price", "price_per_carat", "volume_mm3"]].head()

## 8. Grouping and summarizing

The single most useful pandas operation: **split** by a category, **apply** an aggregation, **combine** into a summary.


In [ ]:
# Average price per cut category
df_clean.groupby("cut", observed=True)["price"].mean().sort_values(ascending=False)

Surprising! "Ideal" cuts have the *lowest* average price. That's not because they're worth less — it's because most Ideal-cut diamonds are smaller (carat affects price far more than cut). This is the kind of insight wrangling enables but a chart will make obvious in Lab 4.


In [ ]:
# Multiple aggregations at once
df_clean.groupby("cut", observed=True).agg(
    n=("price", "size"),
    avg_carat=("carat", "mean"),
    avg_price=("price", "mean"),
    median_price=("price", "median"),
).round(2)

Two categories at once — cross-tabulation:


In [ ]:
(df_clean.groupby(["cut", "color"], observed=True)["price"]
        .mean()
        .unstack()
        .round(0))

## 9. A quick visual sanity check

We'll go much deeper on visualization in Lab 4 — but a single chart often catches issues a summary table hides.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_clean["carat"], df_clean["price"], alpha=0.1, s=5)
ax.set_xlabel("Carat")
ax.set_ylabel("Price (USD)")
ax.set_title("Diamond price vs. carat (after wrangling)")
plt.show()

The relationship is clearly non-linear and there's heteroscedasticity (variance grows with carat). Wrangling produced a dataset we can now actually model.


## Summary

The wrangling workflow you just walked through:

1. **Load** → `pd.read_csv` / `sns.load_dataset`
2. **Inspect** → `shape`, `info()`, `describe()`, `head()`
3. **Check missing** → `isna().sum()`
4. **Check duplicates** → `duplicated().sum()`, `drop_duplicates()`
5. **Check impossible values** → boolean filters
6. **Filter and select** → `df[mask]`, `df[cols]`
7. **Derive new columns** → arithmetic on existing columns
8. **Summarize** → `groupby().agg()`

Every dataset for the rest of the semester will start with some subset of these steps.

Next: open **`lab_03_exercise.ipynb`** and apply this workflow to the Google Play Store dataset.
